# Chapitre 3 — Conditions

**420-SN1-RE · Programmation en sciences · Semaine 3**

> **Avant tout :** fais **Fichier → Enregistrer une copie dans Drive** et travaille dans ta copie.

## À faire avant le cours (environ 3 h)

1. Lis ce chapitre en exécutant chaque exemple (environ 1 h).
2. Fais les **exercices préparatoires 3.1 à 3.10** (environ 1 h 30).
3. Dépose ton notebook sur LÉA avant le cours.

## Objectifs

1. Écrire des expressions logiques avec les opérateurs de comparaison et `and`, `or`, `not`.
2. Choisir entre `if`, `if`/`else` et `if`/`elif`/`else`.
3. Ordonner correctement une suite de conditions.
4. Comparer des nombres réels avec une tolérance.
5. Construire un jeu d'essais qui passe par **chaque branche** et teste les **bornes**.

## Rappel
L'**évaluation 1** a lieu à la semaine 4. Elle porte sur les chapitres 1 à 3.

## Problème d'ouverture

Ton labo de chimie mesure le pH de 30 solutions. Pour chacune, il faut indiquer si elle est acide, neutre ou basique, et signaler les mesures impossibles, causées par une sonde mal calibrée.

Jusqu'ici, tes programmes exécutaient toutes leurs instructions, de haut en bas. Pour classer une solution, le programme doit **choisir** quoi faire selon la valeur. C'est le rôle des conditions.

**Exécute d'abord la cellule suivante** : elle prépare l'outil de vérification.

In [ ]:
# Outil de vérification — exécute cette cellule une fois, au début. Ne pas modifier.
import hashlib, math

def _empreinte(v):
    if isinstance(v, bool) or v is None:
        texte = repr(v)
    elif isinstance(v, (int, float)):
        texte = f"{round(float(v), 4):.4f}"
        if texte == "-0.0000":
            texte = "0.0000"
    else:
        texte = repr(v)
    return hashlib.sha256(texte.encode()).hexdigest()[:12]

def verifier(obtenu, empreinte, nom="réponse"):
    if obtenu is None:
        print(f"⏳ {nom} : pas encore de réponse.")
    elif _empreinte(obtenu) == empreinte:
        print(f"✅ {nom} : correct ({obtenu!r})")
    else:
        print(f"❌ {nom} : pas encore. Tu as obtenu {obtenu!r}. Vérifie ton calcul, puis réessaie.")

## 1. Les expressions logiques

Une comparaison donne un `bool` : `True` ou `False`.

| Opérateur | Signification | Exemple | Résultat |
|---|---|---|---|
| `==` | égal à | `7 == 7` | `True` |
| `!=` | différent de | `7 != 7` | `False` |
| `<`, `<=` | plus petit, plus petit ou égal | `3 <= 3` | `True` |
| `>`, `>=` | plus grand, plus grand ou égal | `3 > 5` | `False` |

**Attention :** `=` affecte une valeur, `==` compare deux valeurs. C'est la confusion la plus fréquente du chapitre.

In [ ]:
ph = 6.2
print(ph < 7)
print(ph == 7)

### Combiner des conditions

| Opérateur | Vrai quand | Exemple |
|---|---|---|
| `and` | les deux conditions sont vraies | `ph >= 0 and ph <= 14` |
| `or` | au moins une est vraie | `ph < 0 or ph > 14` |
| `not` | inverse la condition | `not (ph < 7)` |

Python accepte aussi les comparaisons enchaînées, comme en mathématiques : `0 <= ph <= 14`.

In [ ]:
ph = 6.2
print(0 <= ph <= 14)
print(ph < 0 or ph > 14)

## 2. `if` : exécuter seulement si

```python
if condition:
    instructions exécutées seulement si la condition est vraie
suite du programme
```

Trois détails obligatoires :

1. le **deux-points** `:` à la fin de la ligne `if`;
2. l'**indentation** (4 espaces) des instructions qui dépendent de la condition;
3. le retour à la marge gauche pour la suite, qui s'exécute dans tous les cas.

In [ ]:
temperature = 105   # °C
if temperature > 100:
    print("Attention : l'eau bout à pression normale.")
print("Mesure enregistrée.")

**Essaie-le :** change la température à 25 et exécute de nouveau. Quelle ligne disparaît?

## 3. `if` / `else` : deux chemins

In [ ]:
n = 17
if n % 2 == 0:
    print(n, "est pair")
else:
    print(n, "est impair")

## 4. `if` / `elif` / `else` : plusieurs chemins

Python teste les conditions **dans l'ordre** et exécute **seulement le premier bloc** dont la condition est vraie. Les suivants sont ignorés.

In [ ]:
magnitude = 5.4   # échelle de Richter
if magnitude >= 7:
    categorie = "majeur"
elif magnitude >= 6:
    categorie = "fort"
elif magnitude >= 5:
    categorie = "modéré"
else:
    categorie = "léger"
print(f"Séisme {categorie}")

Pourquoi le `elif magnitude >= 5` n'a-t-il pas besoin de vérifier `magnitude < 6`? Parce qu'on n'arrive à cette ligne **que si** les conditions précédentes étaient fausses. C'est aussi pour cette raison que **l'ordre des conditions compte** : l'exercice 3.8 le montre.

## 5. Piège : comparer des nombres réels

In [ ]:
a = 0.1 + 0.2
print(a)
print(a == 0.3)

Les réels sont stockés avec une précision limitée : `0.1 + 0.2` vaut `0.30000000000000004`. Pour comparer deux réels, on vérifie plutôt que leur écart est **plus petit qu'une tolérance** :

In [ ]:
TOLERANCE = 1e-9
a = 0.1 + 0.2
print(abs(a - 0.3) < TOLERANCE)

## 6. Exemple complet : classer une solution selon son pH

### Planifier

| Question | Réponse |
|---|---|
| Intrant | le pH mesuré (réel) |
| Extrant | un message : « acide », « neutre », « basique » ou « mesure invalide » |
| Traitements | pH hors de [0, 14] : invalide; pH < 7 : acide; pH = 7 : neutre; sinon : basique |

### Le jeu d'essais : une ligne par branche, plus les bornes

Un bon jeu d'essais passe par **chaque branche** du programme au moins une fois, et teste les **bornes**, c'est-à-dire les valeurs où le résultat change.

| pH | Attendu | Ce que ce cas teste |
|---|---|---|
| 3 | acide | branche « acide » |
| 7 | neutre | branche « neutre », et la borne 7 |
| 11 | basique | branche « basique » |
| −1 | mesure invalide | branche « invalide », côté bas |
| 15 | mesure invalide | branche « invalide », côté haut |
| 0 | acide | borne inférieure : 0 est valide |
| 14 | basique | borne supérieure : 14 est valide |

### Coder

In [ ]:
ph = 7.0

if ph < 0 or ph > 14:
    resultat = "mesure invalide"
elif ph < 7:
    resultat = "acide"
elif ph == 7:
    resultat = "neutre"
else:
    resultat = "basique"

print(f"pH {ph} : {resultat}")

### Vérifier
Exécute le programme pour **chacune** des 7 lignes du jeu d'essais. Une seule ligne fausse suffit pour que le programme soit faux.

*Question :* le test `ph == 7` est-il risqué, selon la section 5? Si le pH vient d'un calcul, oui. Ici, il vient d'une mesure saisie : on l'accepte. En classe, on discutera d'une meilleure version.

---
# Exercices préparatoires

Remplace les `None`, exécute, puis lance la vérification. As-tu exécuté la cellule de l'outil de vérification?

### Exercice 3.1 — Vrai ou faux?
Sans exécuter de code, donne la valeur (`True` ou `False`, **sans** guillemets) de chaque expression.

| Variable | Expression |
|---|---|
| `e1` | `3 > 2 and 2 > 5` |
| `e2` | `not (4 == 4)` |
| `e3` | `7 % 2 == 1 or 10 / 2 == 3` |
| `e4` | `0 <= 5 < 10` |

In [ ]:
e1 = None
e2 = None
e3 = None
e4 = None

In [ ]:
# Vérification
verifier(e1, "60a33e6cf515", "e1")
verifier(e2, "60a33e6cf515", "e2")
verifier(e3, "3cbc87c7681f", "e3")
verifier(e4, "3cbc87c7681f", "e4")

### Exercice 3.2 — Pair ou impair
Avec un `if` / `else` et l'opérateur `%`, donne à `parite` la valeur `"pair"` ou `"impair"`.

In [ ]:
n = 17
parite = None
# ta condition ici
print(parite)

In [ ]:
# Vérification
verifier(parite, "5861d6642f2b", "parite")

### Exercice 3.3 — Valeur absolue, sans `abs()`
Calcule la valeur absolue de `x` avec une condition, **sans** utiliser `abs()`.

In [ ]:
x = -4.2
valeur = None
# ta condition ici
print(valeur)

In [ ]:
# Vérification
verifier(valeur, "ee873c079d6f", "valeur")

### Exercice 3.4 — L'état de l'eau
À pression normale, on adopte la convention suivante :

- température de 0 °C ou moins : `"solide"`;
- température de moins de 100 °C : `"liquide"`;
- sinon : `"gaz"`.

Écris la structure `if` / `elif` / `else`, puis teste-la avec 100 °C.

In [ ]:
temperature = 100   # °C
etat = None
# ta structure ici
print(etat)

In [ ]:
# Vérification
verifier(etat, "2b5580c2e508", "etat")

### Exercice 3.5 — Année bissextile
Une année est bissextile si elle est divisible par 4, **sauf** si elle est divisible par 100, **à moins** qu'elle soit aussi divisible par 400. Ainsi, 2024 et 2000 sont bissextiles, mais 1900 ne l'est pas.

Donne à `bissextile` la valeur `True` ou `False`. Une seule expression logique suffit.

In [ ]:
annee = 1900
bissextile = None
print(bissextile)

In [ ]:
# Vérification
verifier(bissextile, "60a33e6cf515", "bissextile")

### Exercice 3.6 — Combien de racines?
Pour l'équation ax² + bx + c = 0, le discriminant Δ = b² − 4ac détermine le nombre de racines réelles : 2 si Δ > 0, 1 si Δ = 0, 0 si Δ < 0.

Donne à `nb_racines` la bonne valeur (un entier).

In [ ]:
a = 1
b = -3
c = 2
nb_racines = None
print(nb_racines)

In [ ]:
# Vérification
verifier(nb_racines, "7a284f128e61", "nb_racines")

### Exercice 3.7 — Comparer des réels
Une simulation calcule la position finale `x = 0.1 * 3` (m). On attend 0,3 m. Calcule :

- `egal_naif` : la comparaison directe avec `==`;
- `egal_tolerant` : la comparaison avec une tolérance de 10⁻⁹.

In [ ]:
x = 0.1 * 3
egal_naif = None
egal_tolerant = None
print(egal_naif, egal_tolerant)

In [ ]:
# Vérification
verifier(egal_naif, "60a33e6cf515", "egal_naif")
verifier(egal_tolerant, "3cbc87c7681f", "egal_tolerant")

### Exercice 3.8 — L'ordre compte
Ce programme doit classer un séisme : 7 et plus, « majeur »; 6 et plus, « fort »; 5 et plus, « modéré »; sinon « léger ». Pour une magnitude de 6,5, il affiche pourtant « modéré ».

Trouve l'erreur de **logique** et corrige-la. Aucun message d'erreur ne t'aidera : c'est ton raisonnement qui trouve l'erreur.

In [ ]:
magnitude = 6.5
if magnitude >= 5:
    categorie = "modéré"
elif magnitude >= 6:
    categorie = "fort"
elif magnitude >= 7:
    categorie = "majeur"
else:
    categorie = "léger"
print(categorie)

In [ ]:
# Vérification
verifier(categorie, "ec58a8306045", "categorie")

### Exercice 3.9 — Le plus grand de trois
Trouve la plus grande des trois mesures **sans** utiliser `max()`.

In [ ]:
a = 4.1
b = 9.3
c = 7.8
plus_grand = None
# ta structure ici
print(plus_grand)

In [ ]:
# Vérification
verifier(plus_grand, "6cbe6f8688f8", "plus_grand")

### Exercice 3.10 — Le spectre électromagnétique
Classe une onde selon sa longueur d'onde λ, en nanomètres :

- moins de 400 nm : `"ultraviolet"`;
- de 400 à 700 nm **inclusivement** : `"visible"`;
- plus de 700 nm : `"infrarouge"`.

Teste avec la borne 700 nm.

In [ ]:
longueur_onde = 700   # nm
domaine = None
# ta structure ici
print(domaine)

In [ ]:
# Vérification
verifier(domaine, "5262fdc68654", "domaine")

---
# Laboratoire 3 (en classe)

Pour chaque exercice : fiche de planification avec un jeu d'essais qui passe par **chaque branche**, code, vérification.

### Labo 3.1 (essentiel) — Les racines d'une équation du second degré
Écris un programme complet pour ax² + bx + c = 0 :

- si a = 0, l'équation n'est pas du second degré : afficher un message;
- si Δ > 0 : deux racines, r1 = (−b + √Δ) / 2a et r2 = (−b − √Δ) / 2a;
- si Δ = 0 : une racine;
- si Δ < 0 : aucune racine réelle.

**Jeu d'essais à compléter avant de coder :** au moins un cas par branche. La vérification ci-dessous teste le cas a = 1, b = −3, c = 2.

In [ ]:
import math
a = 1
b = -3
c = 2
nb_racines = None
r1 = None
r2 = None
# ton programme ici
print(nb_racines, r1, r2)

In [ ]:
# Vérification
verifier(nb_racines, "7a284f128e61", "nb_racines")
verifier(r1, "7a284f128e61", "r1")
verifier(r2, "64e4b892473f", "r2")

### Labo 3.2 (intermédiaire) — Quel triangle?
Avec trois longueurs, un triangle n'existe que si chaque côté est plus petit que la somme des deux autres (inégalité triangulaire). Donne à `type_triangle` la valeur `"invalide"`, `"équilatéral"`, `"isocèle"` ou `"scalène"`.

**Jeu d'essais :** prévois au moins un cas par résultat. La vérification teste 5, 5, 8.

In [ ]:
a = 5
b = 5
c = 8
type_triangle = None
# ton programme ici
print(type_triangle)

In [ ]:
# Vérification
verifier(type_triangle, "4096d17f8c81", "type_triangle")

### Labo 3.3 (défi) — Est-il rectangle?
Ajoute à ton programme un test qui indique si le triangle est **rectangle** (Pythagore). Les côtés peuvent être des réels issus de mesures : utilise une tolérance relative de 10⁻⁶, et pense que l'hypoténuse n'est pas forcément `c`.

La vérification teste les côtés 0,6, 0,8 et 1,0, en désordre.

In [ ]:
a = 1.0
b = 0.6
c = 0.8
rectangle = None
# ton programme ici
print(rectangle)

In [ ]:
# Vérification
verifier(rectangle, "3cbc87c7681f", "rectangle")

---
## Pièges fréquents

- Confondre `=` (affectation) et `==` (comparaison).
- Oublier le `:` ou l'indentation après `if`, `elif`, `else`.
- Mettre les conditions d'un `elif` dans le mauvais ordre : le premier bloc vrai l'emporte.
- Comparer des réels avec `==`.
- Écrire `if ph < 0 or > 14` : chaque côté du `or` doit être une comparaison complète, `ph < 0 or ph > 14`.
- Tester une seule branche et conclure que « ça marche ».

## Autoquiz

1. Que vaut `not (3 > 2 or 1 > 2)`?
2. Dans un `if` / `elif` / `else`, combien de blocs peuvent s'exécuter au maximum?
3. Pourquoi `0.1 + 0.2 == 0.3` est-il faux?
4. Combien de cas au minimum faut-il dans le jeu d'essais d'une structure à 4 branches?
5. Qu'est-ce qu'une borne, et pourquoi la tester?

## Vocabulaire
**booléen** · **expression logique** · **opérateur de comparaison** · **opérateur logique** · **condition** · **branche** · **bloc** · **indentation** · **structure conditionnelle** · **tolérance** · **borne** · **erreur de logique**

## Remise
**Fichier → Télécharger → Télécharger le fichier .ipynb**, puis dépôt sur LÉA.

## Pour la semaine 4
Lis le chapitre 4 et fais les exercices préparatoires. **L'évaluation 1** a lieu pendant le labo de la semaine 4 et porte sur les chapitres 1 à 3.